# Демо: плагинные оценщики эпиплексии в `episynth`

Этот блокнот показывает, как пользователь выбирает алгоритм оценки эпиплексии по имени,
как добавить свой, и строит графики по результатам прогона. Для скорости используется
крошечная модель и очень малый бюджет на CPU — достаточно, чтобы продемонстрировать,
что всё считается и рисуется.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from episynth.estimators import EpiplexityEstimator, available, build, register
from episynth.generators import GENERATORS
from episynth.model.moirai import MoiraiConfig
from episynth.training import TrainConfig, build_model
from episynth.windows import WindowSpec

## Доступные оценщики

Встроенные алгоритмы уже зарегистрированы в реестре. `build(name)` выбирает их по имени.

In [ ]:
print("Available estimators:", available())

# Выбор по имени
est = build("prequential")
print("Built:", type(est).__name__, "name =", est.name)

## Крошечная конфигурация

Модель ограничена снизу числом голов: `num_heads = d_model // 64`, поэтому берём
`d_model = 64`, один слой и одну компоненту смеси. Это ~60 тыс. параметров.

In [ ]:
model_cfg = MoiraiConfig(
    d_model=64, num_layers=1, max_seq_len=32,
    patch_sizes=(32,), components=("student_t",),
)
spec = WindowSpec(
    series_length=256, context_max=128, horizon=32,
    patch_size=32, context_lengths=(128,),
)

model = build_model(model_cfg, init_seed=0)
print("model parameters:", sum(p.numel() for p in model.parameters()))

## Прогон prequential-оценщика

Очень маленький бюджет (64 окна, 4 на шаг) на CPU. Прогон выполняется через общий
`Trainer` и пишет артефакты в каталог.

In [ ]:
out_dir = Path("runs/demo_prequential")
config = TrainConfig(n_windows=64, batch_size=4, device="cpu", warmup_steps=0,
                     checkpoint_fractions=(), eval_precision="fp32")
generator = GENERATORS["waveform"]()

est = build("prequential")
result = est.run(model, generator, spec, config, out_dir)
print(json.dumps(result, indent=2))

## Графики по результатам прогона

Читаем записанные артефакты `window_loss.f32` и `steps.csv` и рисуем кривую лосса
(площадь над финальным лоссом — prequential-оценка model part).

In [ ]:
loss = np.memmap(out_dir / "window_loss.f32", dtype=np.float32, mode="r", shape=(config.n_windows,))

k = max(1, int(round(0.01 * config.n_windows)))
final = float(loss[-k:].mean())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(loss, lw=1)
ax[0].axhline(final, color="r", ls="--", label=f"final loss ({final:.3f})")
ax[0].fill_between(range(len(loss)), final, loss, where=loss > final, color="C0", alpha=0.3)
ax[0].set_title("Per-window loss (nats)")
ax[0].set_xlabel("window index")
ax[0].legend()

ax[1].bar(["model part", "data part"],
          [result["model_part_bits"], result["data_part_bits"]],
          color=["C0", "C2"])
ax[1].set_title("Epiplexity estimate (bits)")
ax[1].set_ylabel("bits")
plt.tight_layout()
plt.show()

## Добавление собственного оценщика одним модулем

Нужно реализовать контракт `EpiplexityEstimator` (метод `run`) и зарегистрировать его
декоратором `@register`. Остальные модули не меняются.

In [ ]:
@register("demo_constant")
class ConstantEstimator(EpiplexityEstimator):
    """Тривиальный пример: возвращает нулевую оценку без обучения."""
    name = "demo_constant"

    def run(self, model, generator, spec, config, out_dir):
        return {"model_part_bits": 0.0, "data_part_bits": 0.0}


print("Available after registration:", available())
print("Built:", build("demo_constant").run(None, None, None, None, "unused"))